# 002 - Level2

## 2. Data Filtering

### 🧰 Operadores y cláusulas más comunes

- **WHERE**  
  Filtra filas basándose en una o más condiciones.

- **BETWEEN**  
  Obtiene valores que se encuentran dentro de un rango específico.

- **IN**  
  Filtra múltiples valores al mismo tiempo.

- **LIKE**  
  Permite buscar **patrones** dentro de campos de texto.

- **CASE WHEN**  
  Aplica condiciones para crear **nueva lógica** o **columnas derivadas**.


In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## Ejercicio 2.1: El Clásico "Brasil vs El Mundo"

### 🎯 Objetivo

Ver todos los partidos donde **Brasil** fue el **equipo local** (`Home Team Name`) y el partido se jugó en el **año 2014**.

Además, mostrar únicamente las siguientes columnas:

- `Datetime`
- `Stadium`
- `Home Team Name`
- `Away Team Name`
- Goles del equipo local y visitante


```SQL
SELECT
    datetime,
    stadium,
    "Home Team Name",
    "Away Team Name"
FROM worldcup.matches
WHERE LOWER(TRIM("Home Team Name")) = 'brazil' and year = 2014;
```

In [5]:
df_m = df_matches.copy()

df_r = df_m[
    (df_m['home team name'].str.strip().str.lower() == 'brazil') &
    (df_m['year'] == 2014)
]

respuesta = df_r[['datetime','stadium','home team name','away team name']].reset_index(drop=True)

In [8]:
pl_r = pl_matches.filter(
    (pl.col('home team name').str.strip_chars().str.to_lowercase() == 'brazil') &
    (pl.col('year') == 2014)
).select(
    pl.col('datetime'),
    pl.col('stadium'),
    pl.col('home team name'),
    pl.col('away team name')
)

## Ejercicio 2.2: Filtrado por Rangos y Listas (BETWEEN e IN)

### 🎯 Objetivo

Analizar **mundiales específicos** utilizando la tabla `WorldCups`.

El reto consiste en filtrar los torneos que ocurrieron en un **periodo de tiempo determinado** y donde el **campeón** pertenece a un grupo selecto de potencias futbolísticas.

---

### 📌 Condiciones del filtro

- Seleccionar los mundiales ocurridos **entre los años 1970 y 1990** (inclusive).
- Que el **ganador** (`Winner`) sea uno de los siguientes:
  - `'Brazil'`
  - `'Italy'`
  - `'Germany FR'`

---

### 📊 Columnas a mostrar

- `Year`
- `Country` (País anfitrión)
- `Winner` (Ganador)

---

### 💡 Pistas para la resolución

A continuación se muestran los métodos clave que necesitarás en cada tecnología para resolver este ejercicio:

| Herramienta | Filtro de Rango (1970 - 1990) | Filtro de Lista (Ganadores) |
|------------|------------------------------|------------------------------|
| **Postgres** | `Year BETWEEN 1970 AND 1990` | `Winner IN ('Brazil', 'Italy', 'Germany FR')` |
| **Pandas** | `(df['Year'] >= 1970) & (df['Year'] <= 1990)` | `df['Winner'].isin(['Brazil', 'Italy', 'Germany FR'])` |
| **Polars** | `pl.col('Year').is_between(1970, 1990)` | `pl.col('Winner').is_in(['Brazil', 'Italy', 'Germany FR'])` |


```SQL
SELECT
    year,
    country,
    winner
FROM worldcup.cups
WHERE year BETWEEN 1970 AND 1990 AND
      REPLACE(LOWER(TRIM(winner)), 'germany fr', 'germany') in ('brazil','italy','germany');
```

In [14]:
df_c = df_cups.copy()

res = df_c[
    (df_c['winner'].str.strip().str.lower().str.replace('germany fr', 'germany')).isin(['brazil','italy','germany']) &
    (df_c['year'].between(1970,1990))
]

res_f = res[['year','country','winner']].reset_index(drop=True)

In [13]:
res = pl_cups.filter(
    (pl.col('winner').str.strip_chars().str.to_lowercase().str.replace('germany fr','germany')).is_in(['brazil','italy','germany']) &
    (pl.col('year').is_between(1970,1990))
).select(
    pl.col('year'),
    pl.col('country'),
    pl.col('winner')
)

## Ejercicio 2.3: El Detective de Estadios

### 🎯 Objetivo

En la tabla `WorldCupMatches`, encontrar todos los partidos que se jugaron en un **estadio** cuyo nombre contenga la palabra **"Estadio"** (en español) o **"Stadium"** (en inglés).

---

### 📌 Condiciones

- Filtrar la columna `Stadium` buscando los patrones:
  - `"Estadio"`
  - `"Stadium"`
  
  > Sin importar si aparecen al **inicio**, **medio** o **final** del texto.
- Que la **asistencia** (`Attendance`) sea **mayor a 50,000** personas.

---

### 📊 Columnas a mostrar

- `Year`
- `Stadium`
- `City`
- `Attendance`

---

### 💡 Pistas

- **SQL**  
  ```sql
  WHERE (Stadium LIKE '%Estadio%' OR Stadium LIKE '%Stadium%')


```SQL
SELECT
    year,
    stadium,
    city,
    attendance
FROM worldcup.matches
WHERE (TRIM(LOWER(stadium)) like '%estadio%' OR TRIM(LOWER(stadium)) like '%stadium%')
AND (attendance::INTEGER) > 50000;
```

In [24]:
df_m = df_matches.copy()

df_m['attendance'] = pd.to_numeric(df_m['attendance'], errors='coerce')

res = df_m[
    (df_m['stadium'].str.strip().str.contains('estadio|stadium', case=False, na=False)) &
    (df_m['attendance'] > 50000)
]

res_f = res[['year','stadium','city','attendance']]

In [25]:
res_f = pl_matches.filter(
    (pl.col('stadium').str.contains(r"(?i)estadio|stadium")) &
    (pl.col('attendance').cast(pl.Int64, strict=False) > 50000)
).select([
    'year', 'stadium', 'city', 'attendance'
])

res_f

year,stadium,city,attendance
i64,str,str,i64
1930,"""Estadio Centenario""","""Montevideo """,57735
1930,"""Estadio Centenario""","""Montevideo """,70022
1930,"""Estadio Centenario""","""Montevideo """,72886
1930,"""Estadio Centenario""","""Montevideo """,79867
1930,"""Estadio Centenario""","""Montevideo """,68346
…,…,…,…
2014,"""Estadio Castelao""","""Fortaleza """,60342
2014,"""Estadio Nacional""","""Brasilia """,68551
2014,"""Estadio Mineirao""","""Belo Horizonte """,58141


## Ejercicio 2.4: Categorización con CASE WHEN

### 🎯 Objetivo

Clasificar los partidos de la tabla `WorldCupMatches` según la **cantidad total de goles**, con el fin de identificar si fueron partidos **aburridos** o **emocionantes**.

---

### 🧠 Lógica de clasificación

- Si el **total de goles** es **0**, la categoría es:
  - `'Sin Goles'`
- Si el **total de goles** es **1 o 2**, la categoría es:
  - `'Bajo Puntaje'`
- Si el **total de goles** es **3 o más**, la categoría es:
  - `'Lluvia de Goles'`

---

### 📌 Condición adicional

- Mostrar **solo los partidos** correspondientes al **Mundial 2014**.


```SQL
SELECT
    "Home Team Name",
    "Away Team Name",
    ("Home Team Goals" + "Away Team Goals") AS total_goals,
    CASE
        WHEN ("Home Team Goals" + "Away Team Goals") = 0 THEN 'Sin Goles'
        WHEN ("Home Team Goals" + "Away Team Goals")  <= 2 THEN 'Bajo Puntaje'
        ELSE 'Lluvia de Goles'
    END AS categoria
FROM worldcup.matches
WHERE year = 2014;
```
##### Alternativa con CTE
```SQL
WITH partidos_con_suma AS (
    -- Aquí hacemos el cálculo una sola vez
    SELECT 
        "Home Team Name", 
        "Away Team Name", 
        year,
        ("Home Team Goals" + "Away Team Goals") AS total_goals
    FROM worldcup.matches
    WHERE year = 2014
)
-- Ahora usamos esa "tabla temporal" para aplicar el CASE WHEN
SELECT 
    "Home Team Name",
    "Away Team Name",
    total_goals,
    CASE 
        WHEN total_goals = 0 THEN 'Sin Goles'
        WHEN total_goals <= 2 THEN 'Bajo Puntaje'
        ELSE 'Lluvia de Goles'
    END AS categoria
FROM partidos_con_suma;
```

In [27]:
df_m = df_matches.copy()

df_2014 = df_m[df_m['year'] == 2014].copy()
df_2014['total_goals'] = df_2014['home team goals'] + df_2014['away team goals']

condiciones = [
    (df_2014['total_goals'] == 0),
    (df_2014['total_goals'] <= 2)
]

opciones = ['Sin goles', 'Bajo Puntaje']

df_2014['categoria'] = np.select(condiciones, opciones, default='Lluvia de Goles')

res_f = df_2014[['home team goals', 'away team goals', 'total_goals', 'categoria']]


In [30]:
res_f = pl_matches.filter(
    pl.col('year') == 2014
).with_columns(
    total_goals = pl.col('home team goals') + pl.col('away team goals')
).with_columns(
    categoria = pl.when(pl.col('total_goals') == 0)
    .then(pl.lit('Sin Goles'))
    .when(pl.col('total_goals') <= 2)
    .then(pl.lit('Bajo Puntaje'))
    .otherwise(pl.lit('Lluvia de Goles'))
).select([
    'home team name','away team name', 'total_goals', 'categoria'
])

## Ejercicio 2.5: El Semáforo de Resultados

### 🎯 Objetivo

Analizar el **desempeño de los equipos locales** en la tabla `WorldCupMatches`.  
Para ello, se debe crear una nueva columna llamada **`Resultado_Local`** que compare los goles del partido.

---

### 🚦 Lógica del resultado

- Si `Home Team Goals` **>** `Away Team Goals` → `'Gana Local'`
- Si `Home Team Goals` **<** `Away Team Goals` → `'Gana Visitante'`
- Si ambos son **iguales** → `'Empate'`

---

### 📌 Condiciones adicionales

- Filtrar únicamente los partidos donde el **equipo local** (`Home Team Name`) sea:
  - `'Argentina'`
  - `'Brazil'`

---

### 📊 Columnas a mostrar

- `Year`
- `Home Team Name`
- `Away Team Name`
- `Resultado_Local`


```SQL
SELECT
    year,
    "Home Team Name",
    "Away Team Name",
    "Home Team Goals",
    "Away Team Goals",
    CASE
        WHEN "Home Team Goals" > "Away Team Goals" THEN 'Gana Local'
        WHEN "Home Team Goals" < "Away Team Goals" THEN 'Gana Visitante'
        ELSE ' Empate'
    END AS resultado
FROM worldcup.matches
WHERE LOWER(TRIM("Home Team Name")) in ('argentina','brazil');
```

In [36]:
df_m = df_matches.copy()

df_m = df_m[df_m['home team name'].str.strip().str.lower().isin(['argentina','brazil'])]

condiciones = [
    (df_m['home team goals'] > df_m['away team goals']),
    (df_m['home team goals'] < df_m['away team goals'])
]

opciones = ['Gana Local', 'Gana Visitante']

df_m['resultado'] = np.select(condiciones, opciones, default='Empate')

res_f = df_m[['home team name','away team name','home team goals','away team goals', 'resultado']]

In [42]:
res_f = pl_matches.filter(
    pl.col('home team name').str.strip_chars().str.to_lowercase().str.contains(r'(?i)argentina|brazil')
).with_columns(
    resultado = pl.when(pl.col('home team goals') > pl.col('away team goals'))
    .then(pl.lit('Gana Local'))
    .when(pl.col('home team goals') < pl.col('away team goals'))
    .then(pl.lit('Gana Visitante'))
    .otherwise(pl.lit('Empate'))
).select([
    'year','home team name', 'away team name', 'home team goals', 'resultado'
])

res_f

year,home team name,away team name,home team goals,resultado
i64,str,str,i64,str
1930,"""Argentina""","""France""",1,"""Gana Local"""
1930,"""Argentina""","""Mexico""",6,"""Gana Local"""
1930,"""Brazil""","""Bolivia""",4,"""Gana Local"""
1930,"""Argentina""","""Chile""",3,"""Gana Local"""
1930,"""Argentina""","""USA""",6,"""Gana Local"""
…,…,…,…,…
2014,"""Argentina""","""Switzerland""",1,"""Gana Local"""
2014,"""Brazil""","""Colombia""",2,"""Gana Local"""
2014,"""Argentina""","""Belgium""",1,"""Gana Local"""
